# NBA Hustle Stats Scraper
**Author:** Dylan Hahami  
**Source:** [NBA.com/stats](https://www.nba.com/stats/players/hustle)

Scrapes per-36 hustle statistics (deflections, contested shots, screen assists, loose balls recovered, box outs) for every NBA season from **2015-16 through 2024-25** using the NBA Stats API.

### Output
- `data/hustle_stats.csv` — combined multi-season hustle stats, one row per player per season

### Notes
- The NBA Stats API is a public endpoint — no authentication required.
- A 1-second delay between requests is included to avoid rate limiting.
- Run cells top to bottom.

## 0. Imports & Setup

In [ ]:
import os
import time
import requests
import pandas as pd

os.makedirs("data", exist_ok=True)

print("Libraries loaded.")

## 1. Configuration

Define the API endpoint, request headers, and the seasons to collect.

**Headers:** The NBA Stats API requires an `origin` and `referer` header to accept requests from outside the browser. No cookies or session tokens are needed.

In [ ]:
BASE_URL = "https://stats.nba.com/stats/leaguehustlestatsplayer"

HEADERS = {
    "accept": "*/*",
    "accept-language": "en-US,en;q=0.9",
    "origin": "https://www.nba.com",
    "referer": "https://www.nba.com/",
    "user-agent": "Mozilla/5.0",
}

# Parameters shared across all seasons
BASE_PARAMS = {
    "College": "", "Conference": "", "Country": "",
    "DateFrom": "", "DateTo": "", "Division": "",
    "DraftPick": "", "DraftYear": "", "GameScope": "",
    "Height": "", "ISTRound": "", "LastNGames": "0",
    "LeagueID": "00", "Location": "", "Month": "0",
    "OpponentTeamID": "0", "Outcome": "", "PORound": "0",
    "PaceAdjust": "N", "PerMode": "Per36",
    "PlayerExperience": "", "PlayerPosition": "",
    "PlusMinus": "N", "Rank": "N",
    "SeasonSegment": "", "SeasonType": "Regular Season",
    "TeamID": "0", "VsConference": "", "VsDivision": "", "Weight": "",
}

# Seasons to collect: 2015-16 through 2024-25
SEASONS = [f"{y}-{str(y + 1)[-2:]}" for y in range(2015, 2025)]
print(f"Seasons to collect ({len(SEASONS)}): {SEASONS}")

## 2. Fetch Hustle Stats — All Seasons

Loop through each season, call the API, and accumulate results. Failed seasons are logged and skipped rather than crashing the loop.

In [ ]:
all_seasons = []
failed_seasons = []

for season in SEASONS:
    params = {**BASE_PARAMS, "Season": season}

    try:
        response = requests.get(BASE_URL, headers=HEADERS, params=params, timeout=15)
        response.raise_for_status()

        data = response.json()

        # Validate expected response structure
        if "resultSets" not in data or not data["resultSets"]:
            raise ValueError(f"Unexpected response structure for season {season}")

        result = data["resultSets"][0]
        df_season = pd.DataFrame(result["rowSet"], columns=result["headers"])
        df_season["SEASON"] = season

        all_seasons.append(df_season)
        print(f"  {season}: {len(df_season)} players fetched")

    except requests.exceptions.HTTPError as e:
        print(f"  {season}: HTTP error — {e}")
        failed_seasons.append(season)
    except requests.exceptions.Timeout:
        print(f"  {season}: Request timed out")
        failed_seasons.append(season)
    except Exception as e:
        print(f"  {season}: Unexpected error — {e}")
        failed_seasons.append(season)

    time.sleep(1)  # Respectful delay between requests

print(f"\nDone. {len(all_seasons)}/{len(SEASONS)} seasons collected.")
if failed_seasons:
    print(f"Failed seasons: {failed_seasons}")

## 3. Combine & Preview

In [ ]:
if not all_seasons:
    raise RuntimeError("No data collected — check your network connection and try again.")

combined_df = pd.concat(all_seasons, ignore_index=True)

print(f"Shape: {combined_df.shape}")
print(f"Seasons present: {sorted(combined_df['SEASON'].unique())}")
print(f"\nColumns: {list(combined_df.columns)}")
combined_df.head()

## 4. Save to CSV

Saving as CSV rather than Excel — lighter, faster, and plays nicer with R and version control.

In [ ]:
output_path = "data/hustle_stats.csv"
combined_df.to_csv(output_path, index=False)
print(f"Saved {len(combined_df)} rows to '{output_path}'")